In [36]:
from pathlib import Path
import pandas as pd

BASE_OUTPUTS_DIR = Path("outputs/")

LANGUAGES = ["acehnese", "javanese", "indonesian", "hitu", "minangkabau", "vietnamese"]
PROMPT_TYPES = ["few_shots_only", "few_shots_and_word_candidates", "topic_related"]

def find_per_model_output_files(lang: str, prompt_type: str) -> list:
    outputs_dir = BASE_OUTPUTS_DIR / prompt_type
    pattern = f"{lang}_{prompt_type}_*.csv"
    files = sorted(outputs_dir.glob(pattern))
    files = [f for f in files if not f.name.endswith(".bak")]
    return files


def load_all_rows() -> pd.DataFrame:
    all_dfs = []

    for prompt_type in PROMPT_TYPES:
        for lang in LANGUAGES:
            files = find_per_model_output_files(lang, prompt_type)

            if not files:
                print(f"[{lang}/{prompt_type}] no files found, skipping")
                continue

            for f in files:
                df = pd.read_csv(f)
                df["language"] = lang
                df["prompt_type"] = prompt_type

                if "model" not in df.columns or "is_thinking_on" not in df.columns:
                    stem = f.stem
                    prefix = f"{lang}_{prompt_type}_"
                    suffix = stem[len(prefix):] if stem.startswith(prefix) else stem
                    if "model" not in df.columns:
                        df["model"] = suffix.rsplit("_thinking_", 1)[0]
                    if "is_thinking_on" not in df.columns:
                        df["is_thinking_on"] = suffix.endswith("_on")

                all_dfs.append(df)

    if not all_dfs:
        raise RuntimeError("No output files found -- check BASE_OUTPUTS_DIR / LANGUAGES / PROMPT_TYPES.")

    combined = pd.concat(all_dfs, ignore_index=True)
    return combined


combined = load_all_rows()
print(f"Loaded {len(combined)} total rows")

combined.head()

Loaded 54000 total rows


,prompt_id,language,length_category,unit,target_length_desc,prompt_type,topic,prompt,output_id,output,...,full_response,model,is_thinking_on,is_valid,fluency_score,cultural_relevance_score,pct_loan_words,seed,candidate_seed,sentence_match_rate
0,b8641c12-b541-45b1-bde2-740053f3baaa,acehnese,5_words,sentence,5 words,few_shots_only,NaN,NaN,8d21c92e-0625-4b69-9aec-825f4a44c2c5,Geudubng meu kawan that.,...,"{""id"": ""msg_011CecP26ARnddtBSBQ7ar7p"", ""contai...",claude-haiku-4-5,False,True,2.0,0.0,0.25,NaN,NaN,NaN
1,a1b43170-baf5-430e-9d58-287c1b3be16e,acehnese,5_words,sentence,5 words,few_shots_only,NaN,NaN,77ebae05-7d89-4416-a10b-7c27b90a2347,Gaki nyoe makan nasi goreng.,...,"{""id"": ""msg_011CecP26BfYdtaQj13QDope"", ""contai...",claude-haiku-4-5,False,True,1.0,4.0,0.20,NaN,NaN,NaN
2,ae51a88a-5190-4e4e-a30b-42c96c3593c3,acehnese,5_words,sentence,5 words,few_shots_only,NaN,NaN,96f96e61-fbd7-4ce8-8421-ae5444f0a964,Gajah nyoe cukôp besar banget.,...,"{""id"": ""msg_011CecP268gvBTPSatj1xHez"", ""contai...",claude-haiku-4-5,False,True,3.0,0.0,0.40,NaN,NaN,NaN
3,4e57ed68-22e6-426d-b1cc-c2cb38751fc0,acehnese,5_words,sentence,5 words,few_shots_only,NaN,NaN,25965002-c1e9-4873-8a50-b19919acd8cf,Geucok meu babah that nyoe.,...,"{""id"": ""msg_011CecP26GNy1W1sr6gbsWgF"", ""contai...",claude-haiku-4-5,False,True,1.0,0.0,0.00,NaN,NaN,NaN
4,3facf64f-3224-4940-a17a-37c9a41559a4,acehnese,5_words,sentence,5 words,few_shots_only,NaN,NaN,a899a39e-8eba-4bc0-a78e-f14d61c3017d,Rumoh meuë gampong that likôt.,...,"{""id"": ""msg_011CecP26CucoV69AQRWhNf2"", ""contai...",claude-haiku-4-5,False,True,1.0,0.0,0.20,NaN,NaN,NaN


In [37]:
import pandas as pd
from pathlib import Path
from scipy.stats import zscore, pearsonr, spearmanr

# ==============================================================
# CONFIGURATION
# ==============================================================

LANGUAGES = [
    "acehnese",
    "javanese",
    "hitu",
    "indonesian",
    "minangkabau",
    "vietnamese",
]

HUMAN_EVAL_DIR = Path("human_fluency_results")

EXPECTED_ROWS = 810
SCORE_COL = "fluency_score"

A1 = "fluency_score_annotator_1"
A2 = "fluency_score_annotator_2"

KEYS = ["language", "prompt_id", "output_id"]

GROUP_COLS = [
    "language",
    "length_category",
    "prompt_type",
]

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ==============================================================
# 1. BUILD FILE PATHS
# ==============================================================

def get_annotator_paths(lang):
    f1 = HUMAN_EVAL_DIR / (
        f"{lang}_stratified_sample_annotator_1 - "
        f"{lang}_stratified_sample_annotator_1.csv"
    )

    f2 = HUMAN_EVAL_DIR / (
        f"{lang}_stratified_sample_annotator_2 - "
        f"{lang}_stratified_sample_annotator_2.csv"
    )

    return f1, f2


# ==============================================================
# 2. SANITY CHECK
# ==============================================================

def sanity_check_annotator(df, filename):

    print("\n" + "=" * 70)
    print(f"Sanity check: {filename}")
    print("=" * 70)

    passed = True

    # Row count
    n_rows = len(df)
    check = n_rows == EXPECTED_ROWS
    print(f"Rows: {n_rows} | Expected: {EXPECTED_ROWS} | PASS: {check}")
    passed &= check

    # Required columns
    required_cols = ["prompt_id", "output_id", SCORE_COL]
    missing_cols = [c for c in required_cols if c not in df.columns]

    check = not missing_cols
    print(f"Required columns present: {check} | Missing: {missing_cols}")
    passed &= check

    if not check:
        return False

    # Missing values
    for col in required_cols:
        n_missing = df[col].isna().sum()
        check = n_missing == 0

        print(f"{col}: missing={n_missing} | PASS: {check}")
        passed &= check

    # Duplicate pairs
    n_duplicates = df.duplicated(
        subset=["prompt_id", "output_id"]
    ).sum()

    check = n_duplicates == 0

    print(
        f"Duplicate pairs: {n_duplicates} | PASS: {check}"
    )
    passed &= check

    # Numeric scores
    scores = pd.to_numeric(df[SCORE_COL], errors="coerce")

    n_non_numeric = scores.isna().sum()
    check = n_non_numeric == 0

    print(
        f"Non-numeric scores: {n_non_numeric} | PASS: {check}"
    )
    passed &= check

    if n_non_numeric == 0:

        # Range check
        n_out_of_range = (~scores.between(1, 5)).sum()
        check = n_out_of_range == 0

        print(
            f"Scores outside 1-5: {n_out_of_range} | PASS: {check}"
        )
        passed &= check

        # Integer check
        n_non_integer = (scores % 1 != 0).sum()
        check = n_non_integer == 0

        print(
            f"Non-integer scores: {n_non_integer} | PASS: {check}"
        )
        passed &= check

        print("\nScore distribution:")
        print(scores.value_counts().sort_index())

    print("\n" + "-" * 70)
    print(f"OVERALL: {'PASS' if passed else 'FAIL'}")
    print("-" * 70)

    return bool(passed)


# ==============================================================
# 3. LOAD AND VALIDATE ALL LANGUAGES
# ==============================================================

annotator_1_list = []
annotator_2_list = []

for lang in LANGUAGES:

    print(f"\n\n{'#' * 70}")
    print(f"PROCESSING: {lang.upper()}")
    print(f"{'#' * 70}")

    f1, f2 = get_annotator_paths(lang)

    assert f1.exists(), f"Missing file: {f1}"
    assert f2.exists(), f"Missing file: {f2}"

    ann1_raw = pd.read_csv(f1)
    ann2_raw = pd.read_csv(f2)

    check1 = sanity_check_annotator(ann1_raw, f1.name)
    check2 = sanity_check_annotator(ann2_raw, f2.name)

    assert check1 and check2, (
        f"Sanity check failed for {lang}. "
        "Fix the annotator files before proceeding."
    )

    # Verify matching prompt_id + output_id pairs
    ids1 = set(
        zip(ann1_raw["prompt_id"], ann1_raw["output_id"])
    )

    ids2 = set(
        zip(ann2_raw["prompt_id"], ann2_raw["output_id"])
    )

    matching = ids1 & ids2

    print(f"\n{lang.upper()} MATCH CHECK")
    print(f"Annotator 1 pairs: {len(ids1)}")
    print(f"Annotator 2 pairs: {len(ids2)}")
    print(f"Matching pairs: {len(matching)}")
    print(f"Missing from Annotator 2: {len(ids1 - ids2)}")
    print(f"Missing from Annotator 1: {len(ids2 - ids1)}")

    assert ids1 == ids2, (
        f"Annotator pair mismatch for {lang}"
    )

    assert len(matching) == EXPECTED_ROWS

    # Select and rename columns
    ann1 = ann1_raw[
        ["prompt_id", "output_id", SCORE_COL]
    ].rename(
        columns={SCORE_COL: A1}
    )

    ann2 = ann2_raw[
        ["prompt_id", "output_id", SCORE_COL]
    ].rename(
        columns={SCORE_COL: A2}
    )

    # Add language identifier
    ann1["language"] = lang
    ann2["language"] = lang

    annotator_1_list.append(ann1)
    annotator_2_list.append(ann2)


# ==============================================================
# 4. COMBINE ALL ANNOTATOR DATA
# ==============================================================

ann1_all = pd.concat(
    annotator_1_list,
    ignore_index=True
)

ann2_all = pd.concat(
    annotator_2_list,
    ignore_index=True
)

print("\n" + "=" * 70)
print("COMBINED ANNOTATOR DATA")
print("=" * 70)

print("Annotator 1:", len(ann1_all))
print("Annotator 2:", len(ann2_all))

print("\nRows per language:")
print(ann1_all["language"].value_counts().sort_index())


# ==============================================================
# 5. MERGE BOTH ANNOTATORS
# ==============================================================

ann = ann1_all.merge(
    ann2_all,
    on=KEYS,
    how="inner",
    validate="one_to_one"
)

print("\nRows scored by both annotators:", len(ann))


# ==============================================================
# 6. Z-SCORE EACH ANNOTATOR WITHIN EACH LANGUAGE
# ==============================================================

ann["z_" + A1] = (
    ann.groupby("language")[A1]
    .transform(lambda x: zscore(x, ddof=0))
)

ann["z_" + A2] = (
    ann.groupby("language")[A2]
    .transform(lambda x: zscore(x, ddof=0))
)

# Average of two raw scores
ann["fluency_mean_2ann"] = ann[
    [A1, A2]
].mean(axis=1)

# Average of two z-scores
ann["fluency_z_2ann"] = ann[
    ["z_" + A1, "z_" + A2]
].mean(axis=1)


# ==============================================================
# 7. MERGE ANNOTATIONS INTO COMBINED
# ==============================================================

ann_cols = [
    c for c in ann.columns
    if c not in KEYS
]

collisions = [
    c for c in ann_cols
    if c in combined.columns
]

print("\nColliding columns:", collisions)

combined_clean = combined.drop(
    columns=collisions
)

annotated = combined_clean.merge(
    ann,
    on=KEYS,
    how="inner",
    validate="m:1"
)

print("\nCombined rows:", len(combined))
print("Annotated rows:", len(annotated))

# Check annotation pairs not found in combined
missing = ann.merge(
    combined[KEYS].drop_duplicates(),
    on=KEYS,
    how="left",
    indicator=True
)

print(
    "Annotated pairs not found in combined:",
    (missing["_merge"] == "left_only").sum()
)



######################################################################
PROCESSING: ACEHNESE
######################################################################

Sanity check: acehnese_stratified_sample_annotator_1 - acehnese_stratified_sample_annotator_1.csv
Rows: 810 | Expected: 810 | PASS: True
Required columns present: True | Missing: []
prompt_id: missing=0 | PASS: True
output_id: missing=0 | PASS: True
fluency_score: missing=0 | PASS: True
Duplicate pairs: 0 | PASS: True
Non-numeric scores: 0 | PASS: True
Scores outside 1-5: 0 | PASS: True
Non-integer scores: 0 | PASS: True

Score distribution:
fluency_score
1    132
2     90
3     92
4    190
5    306
Name: count, dtype: int64

----------------------------------------------------------------------
OVERALL: PASS
----------------------------------------------------------------------

Sanity check: acehnese_stratified_sample_annotator_2 - acehnese_stratified_sample_annotator_2.csv
Rows: 810 | Expected: 810 | PASS: True
Required

In [ ]:
# ==============================================================
# 8. AGGREGATE RESULTS
# ==============================================================

# --- Group 1: Language and Prompt Type ---
group_cols_1 = ["language", "length_category", "prompt_type"]

fluency_summary_1 = (
    annotated.groupby(group_cols_1, dropna=False)
    .agg(
        n_outputs=("fluency_mean_2ann", "size"),
        # fluency_annotator_1=(A1, "mean"),
        # fluency_annotator_2=(A2, "mean"),
        fluency=("fluency_mean_2ann", "mean"),
    )
    .reset_index()
)

print("\n=== SUMMARY 1: LANGUAGE, LENGTH, & PROMPT TYPE ===")
display(fluency_summary_1)

fluency_summary_1.to_csv(
    RESULTS_DIR / "overall_fluency.csv",
    index=False
)


# --- Group 2: Language, Model, Thinking, and Prompt Type ---
group_cols_2 = [
    "language",
    "model",
    "is_thinking_on",
    "length_category",
    "prompt_type"
]

fluency_summary_2 = (
    annotated.groupby(group_cols_2, dropna=False)
    .agg(
        n_outputs=("fluency_mean_2ann", "size"),
        # fluency_annotator_1=(A1, "mean"),
        # fluency_annotator_2=(A2, "mean"),
        fluency=("fluency_mean_2ann", "mean"),
    )
    .reset_index()
)

print("\n=== SUMMARY 2: LANGUAGE, LENGTH, MODEL, THINKING & PROMPT TYPE ===")
display(fluency_summary_2)

fluency_summary_2.to_csv(
    RESULTS_DIR / "detailed_fluency.csv",
    index=False
)

print(f"\nSaved detailed and summary CSVs to {RESULTS_DIR}/")


=== SUMMARY 1: LANGUAGE, LENGTH, & PROMPT TYPE ===


,language,length_category,prompt_type,n_outputs,fluency_annotator_1,fluency_annotator_2,fluency_mean
0,acehnese,15_words,few_shots_and_word_candidates,90,4.388889,3.388889,3.888889
1,acehnese,15_words,few_shots_only,90,3.488889,2.988889,3.238889
2,acehnese,15_words,topic_related,90,2.988889,2.711111,2.850000
3,acehnese,5_words,few_shots_and_word_candidates,90,4.722222,3.955556,4.338889
4,acehnese,5_words,few_shots_only,90,3.866667,3.411111,3.638889
5,acehnese,5_words,topic_related,90,3.222222,2.922222,3.072222
6,acehnese,paragraph,few_shots_and_word_candidates,90,3.700000,3.188889,3.444444
7,acehnese,paragraph,few_shots_only,90,2.811111,2.666667,2.738889
8,acehnese,paragraph,topic_related,90,2.788889,2.477778,2.633333
9,hitu,15_words,few_shots_and_word_candidates,90,2.566667,3.166667,2.866667



=== SUMMARY 2: LANGUAGE, LENGTH, MODEL, THINKING & PROMPT TYPE ===


,language,model,is_thinking_on,length_category,prompt_type,n_outputs,fluency_annotator_1,fluency_annotator_2,fluency_mean
0,acehnese,claude-haiku-4-5,True,15_words,few_shots_and_word_candidates,30,3.966667,2.700000,3.333333
1,acehnese,claude-haiku-4-5,True,15_words,few_shots_only,30,2.466667,1.833333,2.150000
2,acehnese,claude-haiku-4-5,True,15_words,topic_related,30,1.166667,1.100000,1.133333
3,acehnese,claude-haiku-4-5,True,5_words,few_shots_and_word_candidates,30,4.466667,3.366667,3.916667
4,acehnese,claude-haiku-4-5,True,5_words,few_shots_only,30,2.533333,2.033333,2.283333
...,...,...,...,...,...,...,...,...,...
157,vietnamese,gpt-5.4-mini,True,5_words,few_shots_only,30,4.733333,4.966667,4.850000
158,vietnamese,gpt-5.4-mini,True,5_words,topic_related,30,4.500000,4.633333,4.566667
159,vietnamese,gpt-5.4-mini,True,paragraph,few_shots_and_word_candidates,30,3.233333,2.866667,3.050000
160,vietnamese,gpt-5.4-mini,True,paragraph,few_shots_only,30,4.300000,4.533333,4.416667



Saved detailed and summary CSVs to results/


In [39]:
# ==============================================================
# 9. INDIVIDUAL ANNOTATOR SUMMARIES
# ==============================================================

metadata = combined[
    KEYS + GROUP_COLS[1:]
].drop_duplicates()

# Annotator 1
ann1_grouped = ann1_all.merge(
    metadata,
    on=KEYS,
    how="inner",
    validate="one_to_one"
)

mean_ann1 = (
    ann1_grouped
    .groupby(GROUP_COLS, dropna=False)
    .agg(
        n=(A1, "size"),
        fluency_mean=(A1, "mean")
    )
    .reset_index()
)

# Annotator 2
ann2_grouped = ann2_all.merge(
    metadata,
    on=KEYS,
    how="inner",
    validate="one_to_one"
)

mean_ann2 = (
    ann2_grouped
    .groupby(GROUP_COLS, dropna=False)
    .agg(
        n=(A2, "size"),
        fluency_mean=(A2, "mean")
    )
    .reset_index()
)

# ==============================================================
# 10. PEARSON AND SPEARMAN CORRELATION PER LANGUAGE
# ==============================================================

correlation_results = []

for lang, group in ann.groupby("language"):

    # Pearson
    r_pearson, p_pearson = pearsonr(
        group[A1],
        group[A2]
    )

    r_pearson_z, p_pearson_z = pearsonr(
        group["z_" + A1],
        group["z_" + A2]
    )

    # Spearman
    r_spearman, p_spearman = spearmanr(
        group[A1],
        group[A2]
    )

    r_spearman_z, p_spearman_z = spearmanr(
        group["z_" + A1],
        group["z_" + A2]
    )

    correlation_results.append({
        "language": lang,
        "n": len(group),

        "pearson_r": r_pearson,
        "pearson_p": p_pearson,

        "pearson_z_r": r_pearson_z,
        "pearson_z_p": p_pearson_z,

        "spearman_rho": r_spearman,
        "spearman_p": p_spearman,

        "spearman_z_rho": r_spearman_z,
        "spearman_z_p": p_spearman_z,
    })


correlation_df = pd.DataFrame(correlation_results)

print("\n=== CORRELATION SUMMARY ===")
print(correlation_df.to_string(index=False))

correlation_df.to_csv(
    RESULTS_DIR / "inter_annotator_correlations.csv",
    index=False
)

print("\n" + "=" * 70)
print("ALL LANGUAGES PROCESSED SUCCESSFULLY")
print("=" * 70)

print(f"Results saved to: {RESULTS_DIR.resolve()}")


=== CORRELATION SUMMARY ===
   language   n  pearson_r     pearson_p  pearson_z_r   pearson_z_p  spearman_rho    spearman_p  spearman_z_rho  spearman_z_p
   acehnese 810   0.773983 1.613243e-162     0.773983 1.613243e-162      0.771445 8.263477e-161        0.771445 8.263477e-161
       hitu 810   0.466378  5.535206e-45     0.466378  5.535206e-45      0.494350  3.860660e-51        0.494350  3.860660e-51
 indonesian 810   0.564182  2.930287e-69     0.564182  2.930287e-69      0.242606  2.581015e-12        0.242606  2.581015e-12
   javanese 810   0.364778  6.755386e-27     0.364778  6.755386e-27      0.331078  3.579537e-22        0.331078  3.579537e-22
minangkabau 810   0.747115 1.791814e-145     0.747115 1.791814e-145      0.735509 9.194628e-139        0.735509 9.194628e-139
 vietnamese 810   0.553714  2.893157e-66     0.553714  2.893157e-66      0.515866  2.799874e-56        0.515866  2.799874e-56

ALL LANGUAGES PROCESSED SUCCESSFULLY
Results saved to: /Users/michaelsinanta/Documents/4